In [1]:
import sys
# sys.path에 상위 폴더(..)를 추가합니다.
sys.path.append("..")
from psycopg2 import Error
from src.voltaflow.connectors.minio import MinioConnector
import io
import pandas as pd
from src.voltaflow.connectors.db import DbConnector
import os
import time
import json

with open(r"..\assets\chCodeMAP.json") as f:
    chCodeDict = json.load(f)
            
def scan_minio(minio_connector):
    rows_to_insert = []
    paginator = minio_connector.client.get_paginator('list_objects_v2') 
    pages = paginator.paginate(Bucket="tos-stepend")
    
    for idx, page in enumerate(pages):
        if 'Contents' in page:
            for row in page['Contents']:
                if True: #<-TOS에서 데이터 다운로드 시작을 시작한 시간 뒤에 last modeified가 있으면서, required_downlod 컬럼이 True일 때! logic 추가 필요. 우선은 그냥 모든 파일을 가져오도록 합니다.
                    rows_to_insert.append(row['Key'])
    return rows_to_insert
    
def parquet_to_df(minio_connector, minio_path, cell_id=None, exp_id=None):
    res = minio_connector.client.get_object(Bucket="tos-stepend", Key=minio_path)
    df = pd.read_parquet(io.BytesIO(res['Body'].read()))
    df["Cell_ID"] = cell_id
    df["EXP_ID"] = exp_id
    df.columns = [col.lower() for col in df.columns]
    
    return df

def insert_to_db(df, cell_id, exp_id):
    host = os.getenv("DB_HOST")
    database = "mart_db"
    user = os.getenv("DB_USER")
    password = os.getenv("DB_PASSWORD")
    port = os.getenv("DB_PORT")

    try:
        db_conn = DbConnector(host, database, user, password, port)
        db_conn.connect()
        DESTINATION_TABLE_NAME = "exp_fact_tb_new"
        
        db_conn.cursor.execute(f"SELECT column_name FROM information_schema.columns WHERE table_name = '{DESTINATION_TABLE_NAME}'")
        cols = [row[0] for row in db_conn.cursor.fetchall()]
        cols = [x for x in cols if x != 'serial_id']
        
        insert_cols = ', '.join([f'{col}' for col in cols])
        placeholders = ', '.join(['%s'] * len(cols))
        query = f"INSERT INTO {DESTINATION_TABLE_NAME} ({insert_cols}) VALUES ({placeholders})"
        target_data = df.loc[:, cols].values
        db_conn.cursor.executemany(query, target_data)
        db_conn.commit()
        print(exp_id, cell_id, "Data 삽입완료")
    
    except (Exception, Error) as error:
        print(f"데이터베이스 작업 중 오류 발생: {error}")
        if db_conn:
            db_conn.rollback() # 오류 발생 시 롤백
        raise
    
    finally:
        if db_conn:
            db_conn.disconnect()
    

minio_connector = MinioConnector()
rows_to_insert = scan_minio(minio_connector)


Minio connection established successfully


In [2]:
print(len(rows_to_insert))
for minio_path in rows_to_insert:
    print(minio_path)
    cell_id = minio_path.split("/")[0]
    exp_id = minio_path.split("/")[1]
    
    df = parquet_to_df(minio_connector, minio_path, cell_id=cell_id, exp_id=exp_id)
    if 'chcode' in df.columns:
        df['code'] = df['chcode'].map(lambda x: chCodeDict.get(str(x)))
    else:
        print("No 'chcode' or 'code' column found in the dataframe.")
            
    if df.empty:
        print(f"Skipping empty dataframe for {minio_path}")
        continue
    # Extract cell_id and exp_id from the file pat
    
    # Insert data into the database
    insert_to_db(df, cell_id, exp_id)

2194
025CP025CP/EXP_241110_00011/00227540_20241109_JF2_No10_35_025CP025CP_SOC0-100_UDJKF20577.parquet
No 'chcode' or 'code' column found in the dataframe.
데이터베이스 'mart_db'에 성공적으로 연결되었습니다.
EXP_241110_00011 025CP025CP Data 삽입완료
커서가 닫혔습니다.
데이터베이스 연결이 종료되었습니다.
025CP025CP/EXP_241110_00017/00227540_20241109_JF2_No31_25_025CP025CP_SOC0-100_rest_UDJKF12389.parquet
No 'chcode' or 'code' column found in the dataframe.
데이터베이스 'mart_db'에 성공적으로 연결되었습니다.
EXP_241110_00017 025CP025CP Data 삽입완료
커서가 닫혔습니다.
데이터베이스 연결이 종료되었습니다.
025CP025CP/EXP_241110_00019/00227540_20241109_JF2_No31_25_025CP025CP_SOC0-100_rest_UDJKF12686.parquet
No 'chcode' or 'code' column found in the dataframe.
데이터베이스 'mart_db'에 성공적으로 연결되었습니다.
EXP_241110_00019 025CP025CP Data 삽입완료
커서가 닫혔습니다.
데이터베이스 연결이 종료되었습니다.
025CP025CP/EXP_241110_00027/00227540_20241109_JF2_No12_45_025CP025CP_SOC0-100_UDJKF12680.parquet
No 'chcode' or 'code' column found in the dataframe.
데이터베이스 'mart_db'에 성공적으로 연결되었습니다.
EXP_241110_00027 025CP025CP Data 삽입완료
커서가 닫혔습니

In [59]:
minio_path = "BE27B13089/EXP_221117_00130/00218885_20221117_JF1_DV2nd_M_no.9_BE27B13089_SOC65-100_0.25CP_25oC.parquet"
cell_id = minio_path.split("/")[0]
exp_id = minio_path.split("/")[1]

df = parquet_to_df(minio_connector, minio_path, cell_id=cell_id, exp_id=exp_id)